In [1]:
import os
import gc
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV
from collections import defaultdict

In [2]:
print("Hunting for the correct dataset...")
path_1 = "train.parquet"
path_2 = "netflix-prize-data/train.parquet"
correct_path = None

if os.path.exists(path_1) and os.path.getsize(path_1) / 1e6 > 500:
    correct_path = path_1
elif os.path.exists(path_2) and os.path.getsize(path_2) / 1e6 > 500:
    correct_path = path_2

if not correct_path:
    raise FileNotFoundError("Could not find the full 548 MB train.parquet file.")

print(f"Loading data from: {correct_path}")
df_train = pd.read_parquet(correct_path)
df_test = pd.read_parquet(correct_path.replace("train.parquet", "test.parquet"))

Hunting for the correct dataset...
Loading data from: netflix-prize-data/train.parquet


In [6]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_train_sub[['user_id', 'movie_id', 'rating']], reader)

print("\nRunning Grid Search for SVD (This will take a few minutes)...")
# We test 8 different combinations of parameters to find the absolute best fit
param_grid = {
    'n_factors': [50, 100],      # How many latent features to learn
    'n_epochs': [20],            # Iterations over the dataset
    'lr_all': [0.005, 0.01],     # Learning rate
    'reg_all': [0.02, 0.05]      # Regularization (prevents overfitting)
}

# n_jobs=-1 uses all CPU cores to speed up the cross-validation
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1, joblib_verbose=2)
gs.fit(data)

print(f"\nBest Cross-Validation RMSE: {gs.best_score['rmse']:.4f}")
print(f"Winning Parameters: {gs.best_params['rmse']}")


Running Grid Search for SVD (This will take a few minutes)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   6 out of  24 | elapsed:   27.0s remaining:  1.4min
[Parallel(n_jobs=-1)]: Done  19 out of  24 | elapsed:  1.1min remaining:   16.8s



Best Cross-Validation RMSE: 0.9678
Winning Parameters: {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.005, 'reg_all': 0.05}


[Parallel(n_jobs=-1)]: Done  24 out of  24 | elapsed:  1.2min finished


In [7]:
print("\nTraining final SVD model with winning parameters...")
algo = gs.best_estimator['rmse']
trainset = data.build_full_trainset()
algo.fit(trainset)

print("Generating predictions on the test set...")
testset = list(zip(df_test['user_id'], df_test['movie_id'], df_test['rating']))
predictions = algo.test(testset)

final_rmse = accuracy.rmse(predictions, verbose=False)
print(f"Final Test RMSE: {final_rmse:.4f}")


Training final SVD model with winning parameters...
Generating predictions on the test set...
Final Test RMSE: 1.0409


In [8]:
print("\nCalculating MAP@10...")

def get_top_n(predictions, n=10):
    # Map predictions to each user
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est, true_r))
    
    # Sort predictions for each user and retrieve the k highest ones
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

top_10 = get_top_n(predictions, n=10)

aps = [] # Average Precisions
for uid, user_ratings in top_10.items():
    hits = 0
    sum_precs = 0
    for i, (iid, est, true_r) in enumerate(user_ratings):
        if true_r >= 3.5: # The project's relevance threshold
            hits += 1
            sum_precs += hits / (i + 1.0)
            
    if hits > 0:
        aps.append(sum_precs / min(len(user_ratings), 10))
    else:
        aps.append(0.0)

map_10 = np.mean(aps)

print("\n=======================================================")
print("  PHASE 2 COMPLETE — CORRECTED BASELINE RESULTS")
print("=======================================================")
print(f"  Final SVD RMSE   : {final_rmse:.4f}")
print(f"  Final SVD MAP@10 : {map_10:.4f} ({map_10 * 100:.2f}%)")
print("=======================================================")


Calculating MAP@10...

  PHASE 2 COMPLETE — CORRECTED BASELINE RESULTS
  Final SVD RMSE   : 1.0409
  Final SVD MAP@10 : 0.5573 (55.73%)
